In [1]:
from pathlib import Path
import re
import json
import pickle

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


/Users/ledionalame/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# IR Assignment 2 — Notebook 03: Dense Retrieval (R3)

Loads the parsed corpus from `data/interim/parsed_patents.jsonl` produced by `parser.py`,
builds overlapping text chunks, embeds them with `anferico/bert-for-patents`, indexes with
FAISS, and supports both free-text and patent-as-query (document-to-document) search.


In [2]:
# Project paths
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim"
RESULTS_DIR  = PROJECT_ROOT / "results" / "dense"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PARSED_PATH = INTERIM_DIR / "parsed_patents.jsonl"
TOP_K = 10

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PARSED_PATH exists:", PARSED_PATH.exists())
print("RESULTS_DIR:", RESULTS_DIR)


PROJECT_ROOT: /Users/ledionalame/Documents/QMUL/Second Semester/Information Retrieval/Assignment 2/Project/IR-CW2
PARSED_PATH exists: True
RESULTS_DIR: /Users/ledionalame/Documents/QMUL/Second Semester/Information Retrieval/Assignment 2/Project/IR-CW2/results/dense


## 1 — Load Parsed Data

Load from `parsed_patents.jsonl` (produced by `parser.py`). No re-parsing needed.


In [3]:
if not PARSED_PATH.exists():
    raise FileNotFoundError(
        f"Parsed data not found at {PARSED_PATH}.\n"
        "Run: python parser.py"
    )

docs_df = pd.read_json(PARSED_PATH, lines=True)
docs_df["ucid"] = docs_df["ucid"].astype(str)

for field in ["title", "abstract", "claims", "description", "ipc"]:
    if field not in docs_df.columns:
        docs_df[field] = ""
    docs_df[field] = docs_df[field].fillna("")

TAG_SPACE_RE = re.compile(r"\s+")

print(f"Loaded {len(docs_df):,} patents")
docs_df[["ucid", "title", "abstract"]].head(3)


Loaded 2,000 patents


,ucid,title,abstract
0,WO-1979000001-A1,APPARATUS FOR DETERMINING THE EPIDERMIC GROUP ...,
1,WO-1979000002-A1,IMPROVEMENTS RELATING TO MEMBRANE ELECTROPHORESIS,
2,WO-1979000005-A1,IMPROVED CLUTCH-BRAKE SYSTEM FOR ROTARY MOWER,


In [4]:
sample_row = docs_df.iloc[0]
print("UCID:     ", sample_row["ucid"])
print("TITLE:    ", sample_row["title"][:200])
print("ABSTRACT: ", sample_row["abstract"][:400])
print("CLAIMS:   ", sample_row["claims"][:400])


UCID:      WO-1979000001-A1
TITLE:     APPARATUS FOR DETERMINING THE EPIDERMIC GROUP OF A LIVING BEING:DRY-PART DRY-PART GREASY-GREASY
ABSTRACT:  
CLAIMS:    REVENDICATIONS 1) Dispositif permettant la définition d'un groupe epidermique, soit : Sec - Mixte Sec - Mixte Gras - Gras d'un être vivant, pour ce faire en chargeant dans un premier temps celui-ci d'un potentiel énergétique et en le déchargeant dans un deuxième temps à travers sa résistance épidermi- que ; celle-ci étant enregistrée et mesurée à l'aide d'un contrôleur énergétique approprié, défin


#3. Dataframe Chunking

In [5]:
def create_overlapping_chunks(text, title, chunk_size=300, overlap=50):

    """Breaks text into overlapping chunks, prepending the Title for context."""
    if not text or not str(text).strip():
        return []

    words = str(text).split()
    chunks = []
    step_size = chunk_size - overlap

    for i in range(0, len(words), step_size):
        chunk_words = words[i : i + chunk_size]
        chunk_text = " ".join(chunk_words)

        # Inject context!
        contextualized_chunk = f"Title: {title} | Text: {chunk_text}"
        chunks.append(contextualized_chunk)

        if i + chunk_size >= len(words):
            break

    return chunks

In [6]:
all_chunks = []
chunk_metadata = []

# to_dict("records") is ~10x faster than iterrows() — avoids creating a pd.Series per row
records = docs_df[["ucid", "abstract", "description", "claims", "title"]].to_dict("records")

for row in tqdm(records, desc="Chunking patents"):
    full_text = TAG_SPACE_RE.sub(
        " ",
        f"{row['abstract']} {row['description']} {row['claims']}"
    ).strip()

    title = row["title"] or "Unknown Title"

    chunks = create_overlapping_chunks(full_text, title=title)

    for chunk_id, chunk_text in enumerate(chunks):
        all_chunks.append(chunk_text)
        chunk_metadata.append({
            "ucid":     row["ucid"],
            "chunk_id": chunk_id,
            "text":     chunk_text
        })

print(f"\nOriginal patents : {len(docs_df):,}")
print(f"Total chunks     : {len(all_chunks):,}")
assert len(all_chunks) == len(chunk_metadata)

Chunking patents: 100%|██████████| 2000/2000 [00:01<00:00, 1025.73it/s]


Original patents : 2,000
Total chunks     : 53,434


#4. Offline Indexing (Facebook AI Similarity Search - FAISS)

In [7]:
!pip install sentence-transformers faiss-cpu

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 1.1 MB/s  0:00:03 eta 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [sentence-transformers]ence-transformers]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [ ]:
import torch
import faiss
from sentence_transformers import SentenceTransformer

# Detect best available device: MPS (Apple Silicon) → CUDA → CPU
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

# Load model on detected device
MODEL_NAME = "anferico/bert-for-patents"
print(f"Loading {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME, device=DEVICE)

# Batch size: MPS benefits from smaller batches to avoid memory pressure
BATCH_SIZE = 64 if DEVICE in ("mps", "cpu") else 256
print(f"Embedding {len(all_chunks):,} chunks (batch_size={BATCH_SIZE})...")

embeddings = model.encode(
    all_chunks,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True   # required for cosine similarity via IndexFlatIP
)

# FAISS index (always on CPU — faiss-cpu does not support MPS/CUDA)
print("Building FAISS IndexFlatIP...")
embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings.astype("float32"))

# Save index and metadata
faiss_path = INTERIM_DIR / "patent_dense_mvp.index"
meta_path  = INTERIM_DIR / "chunk_metadata_mvp.pkl"

faiss.write_index(index, str(faiss_path))
with open(meta_path, "wb") as f:
    pickle.dump(chunk_metadata, f)

print(f"FAISS index  → {faiss_path}  ({index.ntotal:,} vectors)")
print(f"Metadata     → {meta_path}  ({len(chunk_metadata):,} chunks)")
print("Offline indexing complete.")

/Users/ledionalame/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
No sentence-transformers model found with name anferico/bert-for-patents. Creating a new one with mean pooling.


Using device: mps
Loading anferico/bert-for-patents...
Embedding 53,434 chunks (batch_size=64)...


Batches:   4%|▍         | 37/835 [02:35<55:43,  4.19s/it] 

#5. Online Search

In [15]:
# 5. Online Search Function (MaxP Aggregation)

def search_dense_index(query, top_n=10, faiss_k=100):
    """Embeds a query, searches FAISS, and aggregates chunk scores to rank the parent patents."""
    faiss_path = INTERIM_DIR / "patent_dense_mvp.index"
    meta_path  = INTERIM_DIR / "chunk_metadata_mvp.pkl"

    if not faiss_path.exists() or not meta_path.exists():
        raise FileNotFoundError("FAISS index or metadata not found. Run the indexing cell first!")

    index = faiss.read_index(str(faiss_path))
    with open(meta_path, "rb") as f:
        metadata = pickle.load(f)

    query_embedding = model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(np.array(query_embedding, dtype=np.float32), k=faiss_k)

    patent_scores = {}
    for score, idx in zip(distances[0], indices[0]):
        chunk_meta = metadata[idx]
        ucid = chunk_meta['ucid']
        if ucid not in patent_scores or score > patent_scores[ucid]['score']:
            patent_scores[ucid] = {
                'score': float(score),
                'best_chunk_text': chunk_meta['text']
            }

    ranked_patents = sorted(patent_scores.items(), key=lambda item: item[1]['score'], reverse=True)
    return ranked_patents[:top_n]


#5.a) Free Text Search

In [16]:
test_query = "A device with an absorption core made of cellulose wadding for absorbing liquids"

print(f"User Query: '{test_query}'")
print("=" * 70)

results = search_dense_index(test_query, top_n=5, faiss_k=100)

if not results:
    print("No results found. Check if your FAISS index was populated correctly.")
else:
    for rank, (ucid, data) in enumerate(results, 1):
        print(f"Rank {rank} | Patent UCID: {ucid} | MaxP Score: {data['score']:.4f}")
        snippet = data['best_chunk_text'][:250] + "..." if len(data['best_chunk_text']) > 250 else data['best_chunk_text']
        print(f"Matched Text: {snippet}")
        print("-" * 70)


User Query: 'A device with an absorption core made of cellulose wadding for absorbing liquids'
Rank 1 | Patent ID: 1979000008 | MaxP Score: 0.7189
Matched Text: Title: A DEVICE FOR ABSORBING URINE WITH INCONTINENT PERSONS | Text: that the absorption core (9) of the diaper consists of fluffed cellulose wadding, and in that the liquid-perme¬ able material (lθ) consists of non-woven fabric, at least on the side...
----------------------------------------------------------------------
Rank 2 | Patent ID: 2000000115 | MaxP Score: 0.6989
Matched Text: Title: URINE COLLECTOR COLLECTEUR D'URINE | Text: A urine management device (10) according to claim 1 , wherein said wall material comprises 3 layers, wherein said inner surface (18) is a nonwoven layer, said outer surface (17) is said fibrous hydrop...
----------------------------------------------------------------------
Rank 3 | Patent ID: 2000000112 | MaxP Score: 0.6954
Matched Text: Title: A METHOD FOR COLLECTING AND DISPOSING OF HUMAN WAS

In [ ]:
# Save dense retrieval results
dense_results_path = RESULTS_DIR / "dense_free_text_results.csv"

rows = []
for rank, (ucid, data) in enumerate(results, 1):
    match_row = docs_df[docs_df["ucid"] == ucid]
    title = match_row.iloc[0]["title"] if not match_row.empty else ""
    rows.append({
        "rank":         rank,
        "ucid":         ucid,
        "dense_score":  data["score"],
        "title":        title,
        "best_chunk":   data["best_chunk_text"][:200],
    })

import pandas as pd
save_df = pd.DataFrame(rows)
save_df.to_csv(dense_results_path, index=False)
print(f"Dense results saved to: {dense_results_path}")
print(save_df[["rank", "ucid", "dense_score", "title"]].to_string(index=False))


#5.b) Document to Document Search

In [17]:
def construct_query_from_document(query_patent, strategy="chunking"):
    """Extracts the query from a patent document. Always returns a List[str]."""
    assert strategy in ["first_page", "chunking"], "Unknown strategy"

    if strategy == "first_page":
        text = f"{query_patent.get('title', '')} {query_patent.get('abstract', '')}"
        return [text.strip()]

    # strategy == "chunking"
    full_text = f"{query_patent.get('abstract', '')} {query_patent.get('description', '')} {query_patent.get('claims', '')}"
    full_text = TAG_SPACE_RE.sub(" ", full_text).strip()
    title = query_patent.get('title', 'Unknown Title')
    return create_overlapping_chunks(full_text, title)


In [18]:
def search_by_patent_document(query_patent, docs_df, strategy="chunking", top_n=10, faiss_k=100):
    query_chunks = construct_query_from_document(query_patent, strategy=strategy)
    if not query_chunks:
        return []

    raw_ipc    = str(query_patent.get('ipc', ''))
    target_ipc = raw_ipc[:4] if len(raw_ipc) >= 4 else None
    print(f"Input Patent IPC: {target_ipc}")

    faiss_path = INTERIM_DIR / "patent_dense_mvp.index"
    meta_path  = INTERIM_DIR / "chunk_metadata_mvp.pkl"
    index = faiss.read_index(str(faiss_path))
    with open(meta_path, "rb") as f:
        metadata = pickle.load(f)

    query_embeddings = model.encode(query_chunks, normalize_embeddings=True, show_progress_bar=False)
    distances, indices = index.search(np.array(query_embeddings, dtype=np.float32), k=faiss_k)

    patent_scores = {}
    for chunk_idx in range(len(query_chunks)):
        for score, idx in zip(distances[chunk_idx], indices[chunk_idx]):
            chunk_meta = metadata[idx]
            ucid = chunk_meta['ucid']

            if ucid == query_patent['ucid']:
                continue

            if (ucid not in patent_scores) or (score > patent_scores[ucid]['score']):
                patent_scores[ucid] = {
                    'score':               float(score),
                    'best_db_chunk':       chunk_meta['text'],
                    'matched_query_chunk': query_chunks[chunk_idx]
                }

    ranked_candidates = sorted(patent_scores.items(), key=lambda item: item[1]['score'], reverse=True)
    final_results = []

    for ucid, data in ranked_candidates:
        if len(final_results) >= top_n:
            break
        matching_row = docs_df[docs_df['ucid'] == ucid]
        if matching_row.empty:
            continue
        candidate = matching_row.iloc[0]
        final_results.append({
            'ucid':           ucid,
            'score':          data['score'],
            'title':          candidate['title'],
            'ipc':            str(candidate['ipc']),
            'match_reason_db': data['best_db_chunk']
        })

    return final_results


In [19]:
sample_input_patent = docs_df.iloc[42].to_dict()

doc_results = search_by_patent_document(sample_input_patent, docs_df=docs_df, top_n=5)

for rank, res in enumerate(doc_results, 1):
    print(f"Rank {rank} | UCID: {res['ucid']} | Score: {res['score']:.4f} | IPC: {res['ipc']}")
    print(f"Title: {res['title']}")
    print("-" * 50)


Input Patent Query has the following IPC: 2 B4
Rank 1 | ID: 2000001043 | Score: 0.9825 | IPC: 7 H01R 43/16 H01H 11/04
Title: METHOD FOR MAKING A CONTACT ELEMENT PROCEDE DE REALISATION D'UNE PIECE CONTACTEE
--------------------------------------------------
Rank 2 | ID: 1979000041 | Score: 0.9815 | IPC: 2 A63B 3/00
Title: APPLIANCE FOR ASYMMETRICAL BARS FOR SPORT'S USE
--------------------------------------------------
Rank 3 | ID: 1979000170 | Score: 0.9815 | IPC: 2 B65D 5/56
Title: MANUFACTURING PROCESS OF A FOLDED CARDBOARD PACKING COVERED WITH AN IMPERVIOUS SHEET AND PACKING OBTAINED FROM SUCH PROCESS
--------------------------------------------------
Rank 4 | ID: 1979000171 | Score: 0.9811 | IPC: 2 B65D 5/56
Title: PACKING COMPRISING IN ASSOCIATION ONE PART OF CARDBOARD AND ONE PART OF SYNTHETIC MATERIAL
--------------------------------------------------
Rank 5 | ID: 1979000065 | Score: 0.9809 | IPC: 2 A61B 6/14
Title: APPARATUS FOR PANORAMIC RADIOGRAPHY
---------------------------